In [1]:
# import necessary libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio, create_static_tool_filter
from openai import AsyncAzureOpenAI
from IPython.display import Image, display, Markdown
import os, sys
import pandas as pd
from agents import set_tracing_disabled
set_tracing_disabled(True)

load_dotenv(override=True)

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")


AZURE_OPENAI_API_KEY is available


In [3]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

## let's a define a model pointing to Azure deployment
model_openai_GPT_54 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)

In [4]:
"""
On Windows, a stdio MCP Server started from Jupyter kernel weites to a stderr stream with no real life descriptor,
and crashes with io.UnsupportedOperation: fileno. We send the server's stderr to the null device so it always has somewhere real to write,
which lets every cell below use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux doesn't get affected by this.
"""
import functools
import subprocess
import agents.mcp.server 

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

### Part 01: Long-Term Memory

In [7]:
# load our memory-server mcp server for Agent to use 
params = {
    "command": sys.executable,
    "args": [
        "-m",
        "backend.memory_server",
    ]
}

async with MCPServerStdio(
    params=params,
    client_session_timeout_seconds=30,
) as memory_server:
    memory_tools = await memory_server.list_tools()

memory_tools


[Tool(name='create_entities', title=None, description='\n    Create new entities in the knowledge graph. Entities whose name already exists are ignored.\n\n    Args:\n        entities: The entities to create, each with a name, a type and optional observations.\n\n    Returns:\n        The entities that were actually created.\n    Raises:\n        None\n    ', input_schema={'type': 'object', '$defs': {'Entity': {'description': 'A node in the knowledge graph.', 'properties': {'name': {'description': 'Unique name of the entity', 'title': 'Name', 'type': 'string'}, 'entity_type': {'description': 'Type of the entity, e.g. person, company, project', 'title': 'Entity Type', 'type': 'string'}, 'observations': {'description': 'Facts known about the entity', 'items': {'type': 'string'}, 'title': 'Observations', 'type': 'array'}}, 'required': ['name', 'entity_type'], 'title': 'Entity', 'type': 'object'}}, 'properties': {'entities': {'items': {'$ref': '#/$defs/Entity'}, 'title': 'Entities', 'type'

In [8]:
# let's write inputs for our agent
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Hardeep Singh. I am an AI Engineer. I'm learning a course about AI Agents, which includes a incredible MCP protocol."

In [10]:
# load our memory-server mcp server for Agent to use 
params = {
    "command": sys.executable,
    "args": [
        "-m",
        "backend.memory_server",
    ]
}

async with MCPServerStdio(
    params=params,
    client_session_timeout_seconds=30,
) as memory_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model=model_openai_GPT_54,
        mcp_servers=[memory_server,]
    )

    result = await Runner.run(
        starting_agent=agent,
        input=request,
    )

    display(Markdown(result.final_output))



Nice to meet you, Hardeep Singh. I’ve noted that you’re an AI Engineer and that you’re learning an AI Agents course that includes the MCP protocol.

If you’d like, I can also help explain MCP, AI agents, or help you build a learning roadmap.

In [11]:
# let's ask it about ourself.. where it needs to recall information from long-term memory

async with MCPServerStdio(
    params=params,
    client_session_timeout_seconds=30,
) as memory_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model=model_openai_GPT_54,
        mcp_servers=[memory_server,]
    )

    result = await Runner.run(
        starting_agent=agent,
        input="Hey it's Hardeep's Mother here, could you tell me what he is upto currently...?",
    )

    display(Markdown(result.final_output))



I found a match for **Hardeep Singh**.

Currently, he appears to be:
- **Working as an AI Engineer**
- **Learning a course about AI Agents**
- That course **includes the MCP protocol**

If you want, I can also help summarize what that likely means in plain language.

### Part 02: Web Search

In [8]:
# let's load our Qdrant vector database mcp server (local on-disk storage under 6_mcp/qdrant)
tavily_params = {
    "command": sys.executable,
    "args": [
        "-m",
        "backend.tavily_server",
    ]
}

async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60
) as tavily_server:
    tavily_tools = await tavily_server.list_tools()

[tool.name for tool in tavily_tools]

['tavily_search',
 'tavily_extract',
 'tavily_crawl',
 'tavily_map',
 'tavily_research']

In [6]:
tavily_tools

[Tool(name='tavily_search', title=None, description='\n    Search the web in real time and return ranked, LLM-ready results.\n\n    Args:\n        query: The search query.\n        search_depth: "basic" is balanced, "advanced" is most relevant but slower (2 credits), "fast" and "ultra-fast" favour latency.\n        topic: "news" for current events, "finance" for financial topics, otherwise "general".\n        max_results: Maximum number of results, from 0 to 20.\n        time_range: Only return results published or updated within this window back from today.\n        start_date: Only return results after this date, formatted YYYY-MM-DD.\n        end_date: Only return results before this date, formatted YYYY-MM-DD.\n        include_domains: Restrict results to these domains.\n        exclude_domains: Leave out results from these domains.\n        include_answer: Add a short LLM-generated answer to the response.\n        include_raw_content: Add the cleaned full page content to each resu

In [ ]:
instructions="You search the web for information and briefly summarize the takeways."
request=f"Please research the latest news on Nvidia stock price and briefly summarize its outlook. For context, the current datetime is {pd.Timestamp.now()}."
search_only=create_static_tool_filter(allowed_tool_names=["tavily_search"]) # for limiting our agent to use only this tool..

In [16]:
# let's run our agent with tavily-mcp server..
async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60,
    tool_filter=search_only,
) as tavily_web_server:
    web_search_agent = Agent(
        name="tavily-web-search-agent",
        instructions=instructions,
        model=model_openai_GPT_54,
        mcp_servers=[
            tavily_web_server,
            ],
    )
    
    result = await Runner.run(
        starting_agent=web_search_agent,
        input=request,
    )

    display(Markdown(result.final_output))

Here’s the latest NVDA news and outlook, based on current reporting around Nov. 5, 2026:

- **Stock recently softened:** Nvidia fell about **1.3% to ~$202** on Nov. 2, mainly due to a broader **selloff in chip/tech stocks**, not Nvidia-specific bad news.
- **Earnings catalyst ahead:** Nvidia is scheduled to report **Nov. 17, 2026**. Consensus expects **EPS of $2.48** and **revenue of $109.17B**, both sharply higher year over year.
- **Analyst stance remains bullish:** The stock carries a **Buy consensus** with an average price target around **$341.04**, implying substantial upside from recent levels.
- **Recent targets are still strong:** Some firms trimmed targets a bit, but still stayed positive — e.g. **$300–$350** range from major analysts.
- **Trend is mixed short term, constructive long term:** Reports note NVDA is trading near or slightly below its **200-day moving average**, suggesting near-term caution, but the longer-term growth story remains intact.

**Bottom line:**  
NVDA’s **short-term outlook is choppy** because of sector weakness and earnings uncertainty, but the **medium- to long-term outlook remains positive** thanks to strong expected revenue growth, bullish analyst targets, and an upcoming earnings report that could reset expectations.

### Part 03: Agentic RAG

In [5]:
# let's load our Qdrant vector database mcp server (local on-disk storage under 6_mcp/qdrant)
tavily_params = {
    "command": sys.executable,
    "args": [
        "-m",
        "backend.tavily_server",
    ]
}

async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60
) as tavily_server:
    tavily_tools = await tavily_server.list_tools()

[tool.name for tool in tavily_tools]

['tavily_search',
 'tavily_extract',
 'tavily_crawl',
 'tavily_map',
 'tavily_research']

In [6]:
# let's load our vector-db mcp-server and a give check on it's tools
qdrant_params = {
    "command": sys.executable,
    "args": [
        "-m",
        "backend.qdrant_server",
    ]
}

async with MCPServerStdio(
    params=qdrant_params,
    client_session_timeout_seconds=60,
) as qdrant_server:
    qdrant_tools = await qdrant_server.list_tools()
    
qdrant_tools


[Tool(name='qdrant_store', title=None, description="\n    Store information in the Qdrant vector database so it can be found later by meaning.\n    Creates the collection on first use.\n\n    Args:\n        information: The text to remember.\n        metadata: Optional extra JSON data to keep alongside the text, e.g. source or tags.\n        collection_name: Collection to store into; defaults to the server's default collection.\n\n    Returns:\n        A confirmation naming the collection.\n    Raises:\n        RuntimeError: If AZURE_OPENAI_EMBEDDING_DEPLOYMENT is not set.\n        openai.OpenAIError: If the embedding request fails.\n    ", input_schema={'type': 'object', 'properties': {'information': {'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'title': 'Metadata'}, 'collection_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Collection Name'}}

In [9]:
## let's use this mcp-server to get task done via agent
INSTRUCTIONS = """
You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it. 
"""
search_only=create_static_tool_filter(allowed_tool_names=["tavily_search"]) # for limiting our agent to use only this tool..

In [ ]:
# let's run our agent with tavily-mcp and qdrant-mcp server to create some information collections in our vectordb..
async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60,
    tool_filter=search_only,
) as tavily_web_server:
    async with MCPServerStdio(
        params=qdrant_params,
        client_session_timeout_seconds=60,
    ) as qdrant_vectordb_server:

        agent = Agent(
            name="researcher",
            instructions=INSTRUCTIONS,
            model=model_openai_GPT_54,
            mcp_servers=[tavily_web_server, qdrant_vectordb_server]
        )
        result = await Runner.run(
            starting_agent=agent,
            input= "Research the latest news on Nvidia and store the key facts in your knowledge base.",
        )
        display(Markdown(result.final_output))
    


Done — I researched recent Nvidia news and stored the key facts in my knowledge base.

Key points remembered:
- NVDA was trading near its 200-day moving average and mid-range of its 52-week range.
- Recent coverage highlighted a sharp drop followed by a pre-market rebound.
- Jim Cramer said Nvidia was at its cheapest multiple in about a decade, around 16.5x.
- Options-market reporting suggested high implied volatility and a possible big move.
- Nvidia denied reporting about Jensen Huang being targeted in a Chinese spy plot.
- Nvidia announced a $1 billion commitment to U.S. science and quantum computing.
- Google Cloud and NVIDIA announced infrastructure updates tied to Vera Rubin / HGX B300 at GTC Berlin 2026.
- Recent coverage also emphasized strong revenue growth, guidance near $108B, and some margin pressure.

If you want, I can also turn this into a tighter bullet summary or fetch only the most current market-moving headlines.

In [16]:
# let's run our agent again with just qdrant-mcp server access to answer question related to Nvidia..

async with MCPServerStdio(
    params=qdrant_params,
    client_session_timeout_seconds=60,
) as qdrant_vectordb_server:
    agent = Agent(
        name="researcher",
        instructions=INSTRUCTIONS,
        model=model_openai_GPT_54,
        mcp_servers=[qdrant_vectordb_server],
    )
    result = await Runner.run(
        starting_agent=agent,
        input= "Based on your knowledge base, what's the latest on Nvidia..?",
    )
    display(Markdown(result.final_output))



Here’s the latest Nvidia snapshot from my knowledge base:

- **Strong fundamentals, but valuation pressure**
  - Recent coverage says Nvidia’s growth remains very strong, but the stock has faced **valuation pressure**.
  - One report cited **revenue growth of about 106% to $96B** in the latest period.
  - Next-quarter guidance was around **$108B ±2%**, though margins are under pressure from large AI infrastructure commitments.

- **Product / platform updates**
  - Nvidia’s technical blog highlighted **CUDA 13.5** updates.
  - It also mentioned **Vera Rubin GA support** and performance improvements for **AI and HPC workloads**.

- **Stock / market context**
  - The stock was described as trading around the **middle of its 52-week range** and near its **200-day moving average**.
  - One note said it had recently dropped **$9.02 (-4.22%)** before a pre-market rebound.
  - Commentary from Jim Cramer said Nvidia was at its **cheapest multiple in about a decade** (around **16.5x**).
  - Options activity was described as showing **high implied volatility**, suggesting the market expects a big move.

- **Other notable items**
  - Nvidia **denied** a report alleging Jensen Huang was targeted in a Chinese spy plot.
  - Nvidia also announced a **$1B commitment to U.S. science and quantum computing**.
  - Google Cloud highlighted infrastructure announcements tied to **NVIDIA Vera Rubin / HGX B300** at GTC Berlin 2026.

- **Earnings**
  - One source pointed to an upcoming earnings date around **2026-11-17**.

If you want, I can also give you a **plain-English take on whether Nvidia looks bullish or risky right now** based on these notes.